In [1]:
import utils
import numpy as np
import pandas as pd
import os
import re
from matplotlib.pyplot import cm
import matplotlib.pyplot as plt
import seaborn as sns  # Loads in the theme

pd.set_option('future.no_silent_downcasting', True)
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 20)
#pd.set_option('display.width', 1000)

In [2]:
# Make output folders
output_dir = '/Volumes/project/3025011.02/pre-processed/behaviour/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
# Preparation
pilot_analysis = False
plot_range = [0, 6]

In [4]:
# Import dataset
if pilot_analysis:
    raw_output_path = '/Volumes/project/3025011.02/pilot_data/phd_1_task/experiment_output/behavioural_files_pilot_2'
    unique_subject_ids = [802, 803, 804, 805, 806, 808, 809, 811, 813, 814, 815]
    unique_sessions = [1]
else:
    raw_output_path = '/Volumes/project/3025011.02/raw/'
    unique_subject_ids = [5, 10, 14, 15, 16, 20, 22, 23, 25, 28, 29, 37]
    unique_sessions = [1, 2, 3, 4]

combined_results_dataframe = utils.import_functions.main(raw_output_path, unique_subject_ids, unique_sessions, pilot_analysis, remove_reversals=False, binary_output=True)

['/Volumes/project/3025011.02/raw/sub-005/ses-mri01/beh/behavioural_output_sub-005_session-01_dummy_2025-03-07_17-11-43.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri02/beh/behavioural_output_sub-005_session-02_skyra_2025-03-14_11-00-23.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri03/beh/behavioural_output_sub-005_session-03_skyra_2025-03-21_15-48-02.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri04/beh/behavioural_output_sub-005_session-04_skyra_2025-03-28_15-50-17.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri01/beh/behavioural_output_sub-010_session-01_dummy_2025-03-11_11-18-41.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri02/beh/behavioural_output_sub-010_session-02_skyra_2025-03-13_11-21-05.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri03/beh/behavioural_output_sub-010_session-03_skyra_2025-03-19_11-41-24.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri04/beh/behavioural_output_sub-010_session-04_skyra_2025-03-25_15-18-37.csv',

In [5]:
# Print responses to check if WSLS is correctly translated
print(combined_results_dataframe[['subject_id','session','stimuli_type','response','response_one_back','response_two_back','choicestickiness']])

       subject_id  session  stimuli_type response response_one_back  \
0               5        1             2     down              down   
1               5        1             2     down              down   
0               5        1             1       up                up   
1               5        1             1       up                up   
2               5        1             2     down              down   
...           ...      ...           ...      ...               ...   
15506          37        4             1     down              down   
15507          37        4             1     down              down   
15508          37        4             1       up              down   
15509          37        4             2     down                up   
15509          37        4             1       up                up   

      response_two_back  choicestickiness  
0                  down                 1  
1                  down                 1  
0              

### Now add a column with 'trials_since_reversal'

In [6]:
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe['trials_since_reversal'] = (
    combined_results_dataframe.groupby((combined_results_dataframe['probability_condition'] != combined_results_dataframe['probability_condition'].shift()).cumsum())
      .cumcount()
)
print(combined_results_dataframe[['trials_since_reversal','probability_condition']])

       trials_since_reversal  probability_condition
0                          0                     80
1                          1                     80
2                          2                     80
3                          3                     80
4                          4                     80
...                      ...                    ...
15505                      0                     20
15506                      1                     20
15507                      2                     20
15508                      3                     20
15509                      4                     20

[30683 rows x 2 columns]


# Recode existing variables

In [7]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])
# Add drop WSLS here
WSLS_columns = ['WS_subjective', 'LS_subjective', 'WS_objective', 'LS_objective']
combined_results_dataframe_grouped = combined_results_dataframe.groupby(
    ['subject_id', 'session', 'stimuli_type'],
    sort=False
).cumcount()

# cumcount()==0 is the very first row in each group,
# cumcount()==1 is the second; we want both.
combined_results_dataframe.loc[combined_results_dataframe_grouped.isin([0, 1]), WSLS_columns] = np.nan

print(combined_results_dataframe[['stimuli_type','WS_subjective', 'LS_subjective', 'WS_objective', 'LS_objective']])
combined_results_dataframe = combined_results_dataframe.dropna(subset=WSLS_columns)

       stimuli_type  WS_subjective  LS_subjective  WS_objective  LS_objective
0                 1            NaN            NaN           NaN           NaN
1                 1            NaN            NaN           NaN           NaN
2                 1            1.0            0.0           1.0           0.0
3                 1            1.0            0.0           1.0           0.0
4                 1            1.0            0.0           1.0           0.0
...             ...            ...            ...           ...           ...
15505             2            0.0            0.0           0.0           0.0
15506             2            1.0            0.0           0.0          -1.0
15507             2            0.0           -1.0           0.0          -1.0
15508             2            1.0            0.0           0.0          -1.0
15509             2           -1.0            0.0           0.0           1.0

[30683 rows x 5 columns]


# Now export the data

In [8]:
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'trial'], ascending=[True, True, True])
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_for_r.csv', sep=';', index_label='index')